# 13 - Bono: Delta Sharing (Open Sharing) - proveedor (workspace institucional)

**Objetivo.** Publicar las tablas oro sin copiarlas: un *share* con las cinco tablas del contrato y un destinatario de **Open Sharing** (token) para el workspace de Databricks Free Edition.

**Prerrequisitos.** Metastore con Delta Sharing habilitado y privilegios `CREATE SHARE`, `CREATE RECIPIENT` (los otorga el administrador del workspace institucional).

**Seguridad.** El enlace de activación es una credencial: se entrega por un canal privado al equipo y **nunca** se versiona en Git.

In [0]:
dbutils.widgets.text("catalogo", "oceanwatch_g06")
dbutils.widgets.text("share", "oceanwatch_oro")
dbutils.widgets.text("destinatario", "oceanwatch_free_edition")
C, SHARE, RECIPIENT = (dbutils.widgets.get(k) for k in ("catalogo", "share", "destinatario"))
TABLAS = ["trayectorias_diarias_buque", "estadias_puerto", "puertos_resumen_diario", "trafico_celda_hora", "anomalias_velocidad_diarias"]

In [0]:
spark.sql(f"CREATE SHARE IF NOT EXISTS {SHARE} COMMENT 'Capa oro OceanWatch (contrato oceanwatch-gold v1.0.0)'")
existentes = {r["name"] for r in spark.sql(f"SHOW ALL IN SHARE {SHARE}").collect()}
for t in TABLAS:
    if f"gold.{t}" not in existentes:
        # Historial compartido para lecturas incrementales (CDF no es necesario para el tablero).
        spark.sql(f"ALTER SHARE {SHARE} ADD TABLE {C}.gold.{t} AS gold.{t} WITH HISTORY")
display(spark.sql(f"SHOW ALL IN SHARE {SHARE}"))

name,type,shared_object,added_at,added_by,comment,partitions,history_sharing,cdf_shared,start_version
gold.anomalias_velocidad_diarias,TABLE,oceanwatch_g06.gold.anomalias_velocidad_diarias,2026-11-19T18:01:38.592Z,arangurenmarita@gmail.com,null,null,ENABLED,false,0
gold.estadias_puerto,TABLE,oceanwatch_g06.gold.estadias_puerto,2026-11-19T18:01:36.644Z,arangurenmarita@gmail.com,null,null,ENABLED,false,0
gold.trafico_celda_hora,TABLE,oceanwatch_g06.gold.trafico_celda_hora,2026-11-19T18:01:37.957Z,arangurenmarita@gmail.com,null,null,ENABLED,false,0
gold.trayectorias_diarias_buque,TABLE,oceanwatch_g06.gold.trayectorias_diarias_buque,2026-11-19T18:01:35.684Z,arangurenmarita@gmail.com,null,null,ENABLED,false,0
gold.puertos_resumen_diario,TABLE,oceanwatch_g06.gold.puertos_resumen_diario,2026-11-19T18:01:37.421Z,arangurenmarita@gmail.com,null,null,ENABLED,false,0


In [0]:
# Destinatario de Open Sharing (autenticación por token): consumidor fuera de este metastore.
try:
    spark.sql(
        f"""
        CREATE RECIPIENT IF NOT EXISTS {RECIPIENT}
        COMMENT 'Workspace Databricks Free Edition del equipo'
        """
    )
    spark.sql(f"GRANT SELECT ON SHARE {SHARE} TO RECIPIENT {RECIPIENT}")

    display(spark.sql(f"SHOW GRANTS ON SHARE {SHARE}"))

    # El activation_link aparece en DESCRIBE RECIPIENT: nunca imprimirlo en salida ni subirlo a Git.
    display(
        spark.sql(f"DESCRIBE RECIPIENT {RECIPIENT}").filter(
            "info_name <> 'activation_link'"
        )
    )
except Exception as exc:
    msg = str(exc)
    if "External Delta Sharing is not enabled on the metastore" in msg or "PERMISSION_DENIED" in msg:
        print(
            f"[AVISO] No se pudo crear/asignar el destinatario '{RECIPIENT}' para el share '{SHARE}': "
            "External Delta Sharing no está habilitado en este metastore (restricción de metastore / Free Edition). "
            "El share y sus tablas oro ya quedaron publicados en el paso anterior."
        )
    else:
        raise

[AVISO] No se pudo crear/asignar el destinatario 'oceanwatch_free_edition' para el share 'oceanwatch_oro': External Delta Sharing no está habilitado en este metastore (restricción de metastore / Free Edition). El share y sus tablas oro ya quedaron publicados en el paso anterior.
